# 15. BERTweet Optuna re-tuning + materialization of best-trial metrics

This notebook runs the canonical per-cell Optuna search for the LG-CoTrain pipeline on BERTweet, then materializes per-cell `metrics.json` files from the best trial of each study. The committed result trees produced by this notebook are:

- `results/bertweet/optuna/wge7/{event}/{budget}_set{seed}/trials_10/best_params.json` — Optuna study artifacts (best params + best-trial full metrics)
- `results/bertweet/gpt-4o/optuna-tuned/wge7/{event}/{budget}_set{seed}/metrics.json` — materialized canonical metrics, one per cell

These two trees are the source of the headline LG-CoTrain numbers in the [Reproducing the Results](../README.md#reproducing-the-results) table of the main README.

## What this notebook does

It runs 120 Optuna studies (one per `(event, budget, seed_set)` cell), each with 10 trials, using `vinai/bertweet-base` as the backbone. Each trial trains the full 3-phase pipeline end to end and records its `dev_macro_f1` (Optuna's optimization target) plus the full test metrics (`test_macro_f1`, `test_error_rate`, `test_ece`, `test_per_class_f1`, lambda values, ...) into the trial's `user_attrs` for later extraction.

After all studies finish, a fast in-process post-processing step extracts each study's best-trial test metrics (the trial that won on dev) and writes them as canonical `metrics.json` files at the materialized-tree path. This is bit-exactly equivalent to running a separate "final-run with best params" pass, because training is deterministic on a single machine — same params + same seed + same machine → byte-identical metrics. Skipping that separate pass saves ~2 hours of compute.

## Stages

| Stage | What |
|---|---|
| 1 | Smoke test: 3 studies x 3 trials = 9 BERTweet trainings on a temporary path, sanity-check the pipeline before kicking off the full run |
| 2 | Full Optuna re-tune: 120 studies x 10 trials = 1200 BERTweet trainings (resumable; skips studies whose `best_params.json` already exists) |
| 3 | Materialize tree: extract best-trial metrics from each study and write canonical `metrics.json` files at `results/bertweet/gpt-4o/optuna-tuned/wge7/` |

Stage 2 ties up the kernel for several hours. If Jupyter disconnects, the underlying Python subprocess keeps running and writes results directly to disk, so re-running the cell after reconnect picks up where it left off.


## Setup

In [ ]:
import sys
import subprocess
import time
import json
import shutil
import math
from collections import defaultdict
from pathlib import Path

# Locate the repo root by walking up from the notebook's working directory
def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Cannot find repo root: no ancestor directory contains '{marker}/'")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python:    {sys.version.split()[0]}")

# Branch check
try:
    branch = subprocess.check_output(
        ["git", "branch", "--show-current"],
        cwd=str(repo_root),
        text=True,
    ).strip()
    print(f"Branch:    {branch}")
    if branch != "bertweet-validation":
        print()
        print("WARNING: you are not on the 'bertweet-validation' branch.")
        print("This notebook requires both the Auto* refactor AND the new full_metrics")
        print("user_attr code (commit d613a74). Run `git switch bertweet-validation` first.")
except Exception as e:
    print(f"Could not detect git branch: {e}")

In [ ]:
# Check that GPUs are visible to PyTorch
try:
    import torch
    print(f"torch version:        {torch.__version__}")
    print(f"CUDA available:       {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU detected. The notebook requires at least 1 GPU.")
    N_GPUS = torch.cuda.device_count()
    print(f"CUDA device count:    {N_GPUS}")
    for i in range(N_GPUS):
        props = torch.cuda.get_device_properties(i)
        free_gb, total_gb = (x / 1024**3 for x in torch.cuda.mem_get_info(i))
        print(f"  cuda:{i}  {props.name}  total={total_gb:.1f} GB  free={free_gb:.1f} GB")
except ImportError:
    raise RuntimeError("torch not installed. Activate your training environment first.")

try:
    import transformers
    print(f"transformers version: {transformers.__version__}")
except ImportError:
    raise RuntimeError("transformers not installed. Activate your training environment first.")

try:
    import optuna
    print(f"optuna version:       {optuna.__version__}")
except ImportError:
    raise RuntimeError("optuna not installed. Run `pip install optuna` first.")

print()
print("REMINDER: Disable system sleep before launching the multi-hour Stage 2 cell.")

In [ ]:
# Verify the Auto* refactor AND the new full_metrics user_attr code are in place.
# Both are needed: Auto* lets BERTweet load through the same code path as bert-base,
# and full_metrics lets us materialize tree 3 metrics.json without a separate Phase B2.
import inspect
import lg_cotrain.model as _model_mod
import lg_cotrain.trainer as _trainer_mod
import lg_cotrain.optuna_per_experiment as _optuna_mod

model_src = inspect.getsource(_model_mod)
trainer_src = inspect.getsource(_trainer_mod)
optuna_src = inspect.getsource(_optuna_mod)

checks = [
    ("lg_cotrain.model imports AutoModelForSequenceClassification",
     "AutoModelForSequenceClassification" in model_src),
    ("lg_cotrain.trainer imports AutoTokenizer",
     "AutoTokenizer" in trainer_src),
    ("optuna_per_experiment sets full_metrics user_attr (commit d613a74)",
     "set_user_attr(\"full_metrics\"" in optuna_src),
    ("optuna_per_experiment exposes export_metrics_from_studies helper",
     "def export_metrics_from_studies(" in optuna_src),
    ("optuna_per_experiment accepts model_name parameter",
     "model_name: Optional[str] = None" in optuna_src),
]
all_ok = True
for label, ok in checks:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {label}")
    if not ok:
        all_ok = False
if not all_ok:
    raise RuntimeError(
        "Required code is not in place. Pull the latest bertweet-validation branch "
        "(needs commit d613a74 or later) and re-run this cell."
    )
print()
print("All required code is in place. Safe to proceed.")

In [ ]:
# Output paths:
#   - Optuna studies (intermediate artifact): results/bertweet/optuna/wge7/
#   - Materialized metrics.json (final tree):  results/bertweet/gpt-4o/optuna-tuned/wge7/

OPTUNA_STORAGE = repo_root / "results" / "bertweet" / "optuna" / "wge7"
TREE3_RESULTS = repo_root / "results" / "bertweet" / "gpt-4o" / "optuna-tuned" / "wge7"
DATA_ROOT = repo_root / "data"
MODEL_NAME = "vinai/bertweet-base"
PSEUDO_LABEL_SOURCE = "gpt-4o"
N_TRIALS = 10

OPTUNA_STORAGE.mkdir(parents=True, exist_ok=True)
TREE3_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Optuna studies storage: {OPTUNA_STORAGE}")
print(f"Tree 3 results root:    {TREE3_RESULTS}")
print(f"Data root:              {DATA_ROOT}")
print(f"Model name:             {MODEL_NAME}")
print(f"Pseudo-label source:    {PSEUDO_LABEL_SOURCE}")
print(f"Trials per study:       {N_TRIALS}")
print(f"Num GPUs:               {N_GPUS}")

## Stage 1: smoke test

Runs 3 Optuna studies on `kaikoura_earthquake_2016` (the smallest event), budget=5, seeds 1–3, with `n_trials=3` per study (so 9 BERTweet trainings total). The smoke test writes to a separate `tmp_validation/bertweet_optuna_smoke/` folder so it does not contaminate the canonical tree 3 path. The cleanup cell deletes that folder after the smoke test passes.

**Pass criteria**:
1. All 3 `best_params.json` files exist
2. Each `best_params.json` contains a non-null `best_full_metrics` field with `test_macro_f1` and friends
3. `export_metrics_from_studies` writes 3 valid `metrics.json` files
4. Each materialized `metrics.json` contains `test_macro_f1` between 0.2 and 0.95
5. Multi-GPU dispatch was exercised (timing-based check)

In [ ]:
# Stage 1: smoke test
SMOKE_STORAGE = repo_root / "tmp_validation" / "bertweet_optuna_smoke" / "optuna_studies"
SMOKE_RESULTS = repo_root / "tmp_validation" / "bertweet_optuna_smoke" / "materialized"
SMOKE_STORAGE.mkdir(parents=True, exist_ok=True)
SMOKE_RESULTS.mkdir(parents=True, exist_ok=True)

SMOKE_EVENTS = ["kaikoura_earthquake_2016"]
SMOKE_BUDGETS = [5]
SMOKE_SEEDS = [1, 2, 3]
SMOKE_N_TRIALS = 3   # smaller than full to keep wall clock down
EXPECTED_SMOKE_STUDIES = len(SMOKE_EVENTS) * len(SMOKE_BUDGETS) * len(SMOKE_SEEDS)

cmd = [
    sys.executable, "-m", "lg_cotrain.optuna_per_experiment",
    "--n-trials", str(SMOKE_N_TRIALS),
    "--num-gpus", str(N_GPUS),
    "--storage-dir", str(SMOKE_STORAGE),
    "--data-root", str(DATA_ROOT),
    "--pseudo-label-source", PSEUDO_LABEL_SOURCE,
    "--model-name", MODEL_NAME,
    "--events", *SMOKE_EVENTS,
    "--budgets", *(str(b) for b in SMOKE_BUDGETS),
    "--seed-sets", *(str(s) for s in SMOKE_SEEDS),
]
print("Running smoke test:")
print("  " + " ".join(cmd))
print()
print(f"Expected: {EXPECTED_SMOKE_STUDIES} studies x {SMOKE_N_TRIALS} trials = {EXPECTED_SMOKE_STUDIES * SMOKE_N_TRIALS} BERTweet trainings")
print("=" * 70)

start = time.time()
proc = subprocess.run(cmd, cwd=str(repo_root))
elapsed = time.time() - start

print("=" * 70)
print(f"Wall clock: {elapsed:.0f} seconds = {elapsed/60:.1f} minutes")
if proc.returncode != 0:
    raise RuntimeError(f"Smoke test failed with exit code {proc.returncode}")

# Verify the 3 best_params.json files exist and contain best_full_metrics
smoke_bp_files = []
for ev in SMOKE_EVENTS:
    for b in SMOKE_BUDGETS:
        for s in SMOKE_SEEDS:
            bp = SMOKE_STORAGE / ev / f"{b}_set{s}" / f"trials_{SMOKE_N_TRIALS}" / "best_params.json"
            assert bp.exists(), f"Missing best_params.json: {bp}"
            smoke_bp_files.append(bp)
print(f"Found {len(smoke_bp_files)} best_params.json files (expected {EXPECTED_SMOKE_STUDIES})")

print()
print("Per-study verification (best_full_metrics presence and shape):")
for bp in smoke_bp_files:
    data = json.loads(bp.read_text())
    assert "best_full_metrics" in data, f"Missing best_full_metrics field: {bp}"
    bfm = data["best_full_metrics"]
    assert bfm is not None, f"best_full_metrics is None: {bp}"
    for required in ("test_macro_f1", "test_error_rate", "test_ece", "dev_macro_f1", "test_per_class_f1"):
        assert required in bfm, f"Missing {required} in best_full_metrics: {bp}"
    cell = f"{bp.parent.parent.parent.name}/{bp.parent.parent.name}"
    print(f"  {cell:<40s} dev_f1={bfm['dev_macro_f1']:.4f}  test_f1={bfm['test_macro_f1']:.4f}  test_err={bfm['test_error_rate']:.2f}%  test_ece={bfm['test_ece']:.4f}")

In [ ]:
# Materialize smoke test metrics.json files via export_metrics_from_studies and verify
from lg_cotrain.optuna_per_experiment import export_metrics_from_studies

counts = export_metrics_from_studies(
    storage_dir=str(SMOKE_STORAGE),
    results_root=str(SMOKE_RESULTS),
    n_trials=SMOKE_N_TRIALS,
    events=SMOKE_EVENTS,
    budgets=SMOKE_BUDGETS,
    seed_sets=SMOKE_SEEDS,
)
print(f"export_metrics_from_studies counts: {counts}")
assert counts["written"] == EXPECTED_SMOKE_STUDIES, (
    f"Expected {EXPECTED_SMOKE_STUDIES} metrics.json files written, got {counts['written']}"
)

# Verify each materialized metrics.json has the right shape and reasonable values
smoke_metrics = sorted(SMOKE_RESULTS.glob("*/*/metrics.json"))
assert len(smoke_metrics) == EXPECTED_SMOKE_STUDIES, (
    f"Expected {EXPECTED_SMOKE_STUDIES} metrics.json files, found {len(smoke_metrics)}"
)
print()
print("Per-cell sanity check on materialized metrics.json:")
for f in smoke_metrics:
    m = json.loads(f.read_text())
    assert 0.2 <= m["test_macro_f1"] <= 0.95, f"test_macro_f1 out of range: {f} -> {m['test_macro_f1']}"
    assert 0.0 <= m["test_ece"] <= 1.0, f"test_ece out of range: {f} -> {m['test_ece']}"
    cell = f"{f.parent.parent.name}/{f.parent.name}"
    print(f"  {cell:<40s} test_macro_f1={m['test_macro_f1']:.4f}  test_err={m['test_error_rate']:.2f}%  test_ece={m['test_ece']:.4f}  OK")

# Multi-GPU dispatch check (timing-based)
# If 3 studies ran in parallel on N_GPUS >= 3, total wall clock should be roughly
# the time of one study (the slowest).  If they ran sequentially, total would be ~3x.
import datetime as _dt

def _study_wall_clock_seconds(log_path):
    if not log_path.exists():
        return None
    with open(log_path) as fh:
        lines = fh.readlines()
    if len(lines) < 2:
        return None
    try:
        first = _dt.datetime.strptime(lines[0][:19], "%Y-%m-%d %H:%M:%S")
        last = _dt.datetime.strptime(lines[-1][:19], "%Y-%m-%d %H:%M:%S")
        return (last - first).total_seconds()
    except Exception:
        return None

study_log_files = sorted(SMOKE_STORAGE.glob(f"*/*/trials_{SMOKE_N_TRIALS}/study.log"))
per_study_times = [_study_wall_clock_seconds(p) for p in study_log_files]
per_study_times = [t for t in per_study_times if t is not None]
if per_study_times:
    seq_estimate = sum(per_study_times)
    avg_per_study = sum(per_study_times) / len(per_study_times)
    speedup = seq_estimate / elapsed if elapsed > 0 else 1.0
    print()
    print(f"Multi-GPU dispatch check (timing-based):")
    print(f"  Per-study avg:                 {avg_per_study:.0f}s")
    print(f"  Sum of per-study times:        {seq_estimate:.0f}s")
    print(f"  Actual total wall clock:       {elapsed:.0f}s")
    print(f"  Speedup vs sequential:         {speedup:.2f}x")
    expected_speedup = min(N_GPUS, EXPECTED_SMOKE_STUDIES)
    if N_GPUS == 1:
        print(f"  Single GPU detected: speedup of ~1.0x is expected.")
    elif speedup >= expected_speedup * 0.7:
        print(f"  Speedup is healthy. Multi-GPU dispatch is working.")
    else:
        print(f"  WARNING: speedup is lower than expected ({speedup:.1f}x out of ideal {expected_speedup}x).")
        print(f"  This may be normal for tiny smoke tests where startup overhead dominates.")
        print(f"  If you see this on Stage 2 with 120 studies, investigate before continuing.")

print()
print("STAGE 1 SMOKE TEST COMPLETE.")
print()
print("Key things this validated:")
print("  - Optuna studies run with --model-name vinai/bertweet-base")
print("  - Each best_params.json contains best_full_metrics with all expected fields")
print("  - export_metrics_from_studies materializes metrics.json from best_full_metrics")
print("  - The materialized metrics.json have plausible test_macro_f1 values")
print()
print("Next: run the cleanup cell, then proceed to Stage 2.")

In [ ]:
# Stage 1 cleanup: delete the smoke test folder so Stage 2 starts clean.
smoke_root = repo_root / "tmp_validation" / "bertweet_optuna_smoke"
if smoke_root.exists():
    shutil.rmtree(smoke_root)
    print(f"Deleted {smoke_root}")
else:
    print(f"Already gone: {smoke_root}")

## Stage 2: full Optuna re-tune (120 studies x 10 trials = 1200 BERTweet trainings)

Runs all 10 events x 4 budgets x 3 seeds = 120 Optuna studies in parallel across the available GPUs. Each study runs `n_trials=10` BERTweet trainings, optimizing `dev_macro_f1`, and stores the full per-trial result dict (test metrics + lambdas) in the trial's `user_attrs`.

**Storage**:
- Studies: `results/bertweet/optuna/wge7/{event}/{budget}_set{seed}/trials_10/best_params.json`
- Live progress: `tail -f results/bertweet/optuna/wge7/{event}/{budget}_set{seed}/trials_10/study.log`

**Estimated wall clock**: depends on per-trial training time and GPU count. With 8x H100 and ~3–4 minutes per BERTweet training, 1200 trainings spread across 8 GPUs (work-stealing at the study level) is roughly **5–9 hours**.

**Resumable**: if this cell is interrupted, re-running it will skip studies whose `trials_10/best_params.json` already exists, and will continue partial studies via the incremental-trials replay path. No work is lost.

**No per-event stall**: unlike `lg_cotrain.run_experiment` (which loops events in Python), `lg_cotrain.optuna_per_experiment.run_all_studies` builds ONE flat queue of all 120 studies and dispatches them via `_run_studies_parallel`. Work-stealing means GPUs immediately pick up the next study from the queue when their current one finishes, regardless of which event it belongs to. So the per-event tail-stragglers waste that notebook 14 had does NOT happen here.

In [ ]:
# Stage 2: full grid (~5-9 hours on 8 H100s)
EXPECTED_STUDIES = 120  # 10 events x 4 budgets x 3 seeds

cmd = [
    sys.executable, "-m", "lg_cotrain.optuna_per_experiment",
    "--n-trials", str(N_TRIALS),
    "--num-gpus", str(N_GPUS),
    "--storage-dir", str(OPTUNA_STORAGE),
    "--data-root", str(DATA_ROOT),
    "--pseudo-label-source", PSEUDO_LABEL_SOURCE,
    "--model-name", MODEL_NAME,
    # No --events / --budgets / --seed-sets: defaults to ALL_EVENTS x BUDGETS x SEED_SETS
]
print("Running full retune:")
print("  " + " ".join(cmd))
print()
print(f"Expected: {EXPECTED_STUDIES} studies x {N_TRIALS} trials = {EXPECTED_STUDIES * N_TRIALS} BERTweet trainings")
print("Expected wall clock: ~5-9 hours on 8 H100s")
print("=" * 70)

start = time.time()
proc = subprocess.run(cmd, cwd=str(repo_root))
elapsed = time.time() - start

print("=" * 70)
print(f"Wall clock: {elapsed:.0f} seconds = {elapsed/60:.1f} minutes = {elapsed/3600:.2f} hours")
if proc.returncode != 0:
    print(f"WARNING: subprocess returned exit code {proc.returncode}")
    print("Some studies may have failed. Check best_params.json count below to see how many succeeded.")

# Count best_params.json files written
study_files = sorted(OPTUNA_STORAGE.glob(f"*/*/trials_{N_TRIALS}/best_params.json"))
print()
print(f"best_params.json files written: {len(study_files)} (expected {EXPECTED_STUDIES})")
if len(study_files) < EXPECTED_STUDIES:
    print(f"  MISSING: {EXPECTED_STUDIES - len(study_files)} studies")
    print("  Re-run this cell to retry the missing studies (resume logic will skip the completed ones).")
else:
    print("STAGE 2 COMPLETE: all 120 Optuna studies finished")

## Stage 3: materialize tree 3 metrics.json files

Calls `export_metrics_from_studies(...)` to walk the 120 `best_params.json` files, extract each study's `best_full_metrics` (the full result dict from the trial that won on `dev_macro_f1`), and write them as canonical `metrics.json` files at the tree-3 path:

```
results/bertweet/gpt-4o/optuna-tuned/wge7/{event}/{budget}_set{seed}/metrics.json
```

These files are byte-identical (in shape and values) to what a separate "final-run with best params" pass would have produced, because training is bit-exactly deterministic on a single machine. This is the shortcut that lets us skip Phase B2 entirely.

This cell is fast (seconds, no GPU needed).

In [ ]:
# Stage 3: materialize tree 3 metrics.json from Optuna best_full_metrics
from lg_cotrain.optuna_per_experiment import export_metrics_from_studies

counts = export_metrics_from_studies(
    storage_dir=str(OPTUNA_STORAGE),
    results_root=str(TREE3_RESULTS),
    n_trials=N_TRIALS,
)
print(f"export_metrics_from_studies counts: {counts}")
print()
print(f"  written:          {counts['written']}")
print(f"  skipped:          {counts['skipped']}    (metrics.json already existed; pass overwrite=True to replace)")
print(f"  missing_metrics:  {counts['missing_metrics']}    (best_full_metrics was None - study from before commit d613a74)")
print(f"  missing_study:    {counts['missing_study']}    (no best_params.json found for this cell)")

tree3_files = sorted(TREE3_RESULTS.glob("*/*/metrics.json"))
print()
print(f"Total tree 3 metrics.json files on disk: {len(tree3_files)} (expected {EXPECTED_STUDIES})")
if len(tree3_files) < EXPECTED_STUDIES:
    print(f"  MISSING: {EXPECTED_STUDIES - len(tree3_files)} cells.")
    if counts["missing_metrics"] > 0:
        print(f"  -> {counts['missing_metrics']} studies have best_full_metrics=None.")
        print("     These were probably run before commit d613a74 added the user_attr persistence.")
        print("     Delete those study folders and re-run Stage 2 to regenerate them.")
    if counts["missing_study"] > 0:
        print(f"  -> {counts['missing_study']} studies are missing entirely.")
        print("     Re-run Stage 2 to fill them in (resume logic will skip the completed ones).")
else:
    print("STAGE 3 COMPLETE: all 120 tree 3 metrics.json files materialized.")

## Stage 4: sanity check tree 3

Walks the 120 materialized metrics.json files and verifies:
- All 120 cells exist
- `test_macro_f1` is between 0.2 and 0.95 and not NaN
- All required fields are present (`test_macro_f1`, `test_error_rate`, `test_ece`, `test_per_class_f1`, `dev_macro_f1`, lambda values)
- Print a per-event x budget summary table

In [ ]:
# Stage 4: sanity check the 120 tree 3 metrics
REQUIRED_FIELDS = [
    "test_macro_f1", "test_error_rate", "test_ece", "test_per_class_f1",
    "dev_macro_f1", "lambda1_mean", "lambda2_mean",
]

tree3_files = sorted(TREE3_RESULTS.glob("*/*/metrics.json"))
print(f"Found {len(tree3_files)} metrics.json files at {TREE3_RESULTS}")

issues = []
by_event_budget = defaultdict(list)
for f in tree3_files:
    m = json.loads(f.read_text())

    # Required fields
    for field in REQUIRED_FIELDS:
        if field not in m:
            issues.append(f"{f}: missing field {field}")

    # Range / NaN check
    f1 = m.get("test_macro_f1")
    if f1 is None or (isinstance(f1, float) and math.isnan(f1)):
        issues.append(f"{f}: test_macro_f1 is None or NaN")
    elif not (0.2 <= f1 <= 0.95):
        issues.append(f"{f}: test_macro_f1 out of range (got {f1})")

    # Group for the summary table
    cell_dir = f.parent
    event = cell_dir.parent.name
    budget_seed = cell_dir.name  # "5_set1"
    try:
        budget = int(budget_seed.split("_set")[0])
    except (ValueError, IndexError):
        budget = -1
    by_event_budget[(event, budget)].append(f1)

print()
if issues:
    print(f"WARNING: {len(issues)} sanity check issues:")
    for issue in issues[:20]:
        print(f"  {issue}")
    if len(issues) > 20:
        print(f"  ... and {len(issues) - 20} more")
else:
    print("All sanity checks passed.")

# Per-event x budget summary table
print()
print("=" * 76)
print("Tree 3 (BERTweet + Optuna re-tuned) summary - mean test_macro_f1")
print("=" * 76)
print(f"{'Event':<35s} {'B=5':>8s} {'B=10':>8s} {'B=25':>8s} {'B=50':>8s}")
print("-" * 76)
events_sorted = sorted({ev for ev, _ in by_event_budget})
for ev in events_sorted:
    cells = []
    for b in (5, 10, 25, 50):
        f1s = [x for x in by_event_budget.get((ev, b), []) if x is not None]
        if f1s:
            cells.append(f"{sum(f1s)/len(f1s):>8.4f}")
        else:
            cells.append(f"{'N/A':>8s}")
    print(f"{ev:<35s} {cells[0]} {cells[1]} {cells[2]} {cells[3]}")
print()
print("Per-budget overall mean:")
for b in (5, 10, 25, 50):
    f1s = [
        x for (ev, bb), arr in by_event_budget.items() if bb == b
        for x in arr if x is not None
    ]
    if f1s:
        print(f"  B={b:<3d}  n={len(f1s):>3d}  mean test_macro_f1 = {sum(f1s)/len(f1s):.4f}")